## **UNetConNext model with physics** 

In [1]:
# !pip install the_well[benchmark]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from einops import rearrange
from tqdm import tqdm
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from the_well.benchmark.metrics import VRMSE
from the_well.data import WellDataset
from the_well.benchmark.models.unet_convnext import UNetConvNext

device = "cuda"
base_path = "./datasets"  # path/to/storage

/home/franklin/miniconda3/envs/mlearning/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


### **Develop the model**

In [2]:
class CNextbaseline(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 initial_dimension: int,
                 up_down_blocks: int,
                 blocks_per_stage: int,
                 bottleneck_blocks: int,
                 spatial_resolution: tuple,
                 spatial_dims: int=2):
        
        super().__init__()
        
        self.model = UNetConvNext(
            dim_in = in_channels,
            dim_out = out_channels,
            n_spatial_dims = spatial_dims,
            spatial_resolution = spatial_resolution,
            stages = up_down_blocks,
            blocks_per_stage = blocks_per_stage,
            blocks_at_neck = bottleneck_blocks,
            init_features = initial_dimension 
        )
    
    def forward(self, x):
        return self.model(x)

### **Data and function**

In [3]:
dataset_train = WellDataset(
    well_base_path = f'{base_path}/datasets',
    well_dataset_name = 'active_matter',
    well_split_name = "train",
    n_steps_input = 4,
    n_steps_output = 1,
    use_normalization = False,
)

In [4]:
dataset_valid = WellDataset(
    well_base_path = f'{base_path}/datasets',
    well_dataset_name = 'active_matter',
    well_split_name = "valid",
    n_steps_input = 4,
    n_steps_output = 1,
    use_normalization = False
)

In [5]:
# Remember the key associated to each field
field_names_tmp = dataset_train.field_names

field_names_tmp2 = [
    name for group in field_names_tmp.values() for name in group
]

field_names = {}

for id in range(len(field_names_tmp2)):
    field_names[id] = field_names_tmp2[id]

print(field_names)

{0: 'concentration', 1: 'velocity_x', 2: 'velocity_y', 3: 'D_xx', 4: 'D_xy', 5: 'D_yx', 6: 'D_yy', 7: 'E_xx', 8: 'E_xy', 9: 'E_yx', 10: 'E_yy'}


In [6]:
def concentration_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the concentration field.
    '''
    # Choose the concentration field
    c_id = [k for k, v in field_names.items() if v=='concentration']

    c_list = [] 

    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        c_sample = pred_sample_tmp[c_id[0]]

        c_list.append(c_sample)
    
    return torch.stack(c_list, dim = 0)  

In [7]:
def vel_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the vx and vy components.
    '''
    # Choose the vx and vy components
    vx_id = [k for k, v in field_names.items() if v=='velocity_x']
    vy_id = [k for k, v in field_names.items() if v=='velocity_y']

    vx_list = [] 
    vy_list = []

    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        vx_sample = pred_sample_tmp[vx_id[0]]
        vy_sample = pred_sample_tmp[vy_id[0]]

        vx_list.append(vx_sample)
        vy_list.append(vy_sample)
    
    return torch.stack(vx_list, dim = 0), torch.stack(vy_list, dim = 0)  

In [8]:
Lx = 10 # Lx = Ly = 10 ---> physical grid
dx = Lx/256
dy = dx # Both differentials the same as Lx = Ly
print(dx, dy)

0.0390625 0.0390625


In [9]:
def partials_velocities(prediction_batch, diffx = dx, diffy = dy):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the dvx/dy, dvx/dx and
    dvy/dy, dvy/dx components.

    Input:
    - prediction_batch: tensor of B=4 (To=1 F=11) Lx=256 Ly=256 shape

    Output:
    - 4 tensors of B=4 Lx=256 Ly=256 shape, namely dvxdx, dvxdy, dvydx, and dvydy 
    '''
    # Choose the vx and vy components
    vx_id = [k for k, v in field_names.items() if v=='velocity_x']
    vy_id = [k for k, v in field_names.items() if v=='velocity_y']

    dvxdy_list = [] 
    dvxdx_list = []
    dvydy_list = [] 
    dvydx_list = []

    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        vx_sample = pred_sample_tmp[vx_id[0]]
        vy_sample = pred_sample_tmp[vy_id[0]]

        # Spatial gradients
        dvxdx_tmp, dvxdy_tmp = torch.gradient(vx_sample, spacing = (diffy, diffx))
        dvydx_tmp, dvydy_tmp = torch.gradient(vy_sample, spacing = (diffy, diffx))

        dvxdy_list.append(dvxdy_tmp)
        dvxdx_list.append(dvxdx_tmp)
        dvydy_list.append(dvydy_tmp)
        dvydx_list.append(dvydx_tmp)
    
    return torch.stack(dvxdx_list, dim = 0), torch.stack(dvxdy_list, dim = 0), torch.stack(dvydx_list, dim = 0), torch.stack(dvydy_list, dim = 0) 

In [10]:
def orientation_tensor_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the D_xx, D_xy, D_yx and D_yy components.

    Input:
    - prediction_batch: tensor of B=4 (To=1 F=11) Lx=256 Ly=256 shape

    Output:
    - 4 tensors of B=4 Lx=256 Ly=256 shape, namely Dxx, Dxy, Dyx, and Dyy 
    
    '''
    # Choose the components identification
    Dxx_id = [k for k, v in field_names.items() if v=='D_xx']
    Dxy_id = [k for k, v in field_names.items() if v=='D_xy']
    Dyx_id = [k for k, v in field_names.items() if v=='D_yx']
    Dyy_id = [k for k, v in field_names.items() if v=='D_yy']

    Dxx_list = [] 
    Dxy_list = []
    Dyx_list = [] 
    Dyy_list = []
    
    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        Dxx_sample = pred_sample_tmp[Dxx_id[0]]
        Dxy_sample = pred_sample_tmp[Dxy_id[0]]
        Dyx_sample = pred_sample_tmp[Dyx_id[0]]
        Dyy_sample = pred_sample_tmp[Dyy_id[0]]

        Dxx_list.append(Dxx_sample)
        Dxy_list.append(Dxy_sample)
        Dyx_list.append(Dyx_sample)
        Dyy_list.append(Dyy_sample)
    
    return torch.stack(Dxx_list, dim = 0), torch.stack(Dxy_list, dim = 0), torch.stack(Dyx_list, dim = 0), torch.stack(Dyy_list, dim = 0)

In [11]:
def strain_tensor_comp(prediction_batch):
    ''' 
    Takes the tensor of the shape B (To F) Lx Ly 
    and returns the tensor associated to the E_xx, E_xy, E_yx and E_yy components.

    Input:
    - prediction_batch: tensor of B=4 (To=1 F=11) Lx=256 Ly=256 shape

    Output:
    - 4 strain_rate components for the batch: 4 tensors of B=4 Lx=256 Ly=256 shape,
    namely Exx, Exy, Eyx, and Eyy 
    
    Notes:
    - This function is defined for extracting only the strain tensor components 
    from the prediction batch (batch_size = 4)
    - To is the output time (To = 1) as the output is only one

    Remember: a batch consist of group some items from the training set for 
    improve training velocity. Batch -> minigroups at which training set is splitting
    '''
    # Choose the components identification
    Exx_id = [k for k, v in field_names.items() if v=='E_xx']
    Exy_id = [k for k, v in field_names.items() if v=='E_xy']
    Eyx_id = [k for k, v in field_names.items() if v=='E_yx']
    Eyy_id = [k for k, v in field_names.items() if v=='E_yy']

    Exx_list = [] 
    Exy_list = []
    Eyx_list = [] 
    Eyy_list = []

    for id in range(prediction_batch.shape[0]):
        pred_sample_tmp = prediction_batch[id]

        Exx_sample = pred_sample_tmp[Exx_id[0]]
        Exy_sample = pred_sample_tmp[Exy_id[0]]
        Eyx_sample = pred_sample_tmp[Eyx_id[0]]
        Eyy_sample = pred_sample_tmp[Eyy_id[0]]

        Exx_list.append(Exx_sample)
        Exy_list.append(Exy_sample)
        Eyx_list.append(Eyx_sample)
        Eyy_list.append(Eyy_sample)

    return torch.stack(Exx_list, dim = 0), torch.stack(Exy_list, dim = 0), torch.stack(Eyx_list, dim = 0), torch.stack(Eyy_list, dim = 0)   

## **Procesing**

In [12]:
# Parameters for the dataset splitting into batches
size = 4
workers = 2

train_loader = torch.utils.data.DataLoader(
    dataset=dataset_train,
    shuffle=True,
    batch_size=size,
    num_workers=workers,
    pin_memory = True,
    persistent_workers=True
)

valid_loader = torch.utils.data.DataLoader(
    dataset=dataset_valid,
    shuffle=True,
    batch_size=size,
    num_workers=workers,
    pin_memory = True,
    persistent_workers=True
)

In [13]:
# Normalisation stage
norm_params = torch.load('./model_stats/normalisation_train_param.pt')
mu = norm_params['mu']
sigma = norm_params['sigma']

In [14]:
def preprocess(x):
    return (x - mu) / sigma


def postprocess(x):
    return sigma * x + mu

In [15]:
# Extract the number of fields
F = dataset_train.metadata.n_fields

In [16]:
model_cnet = CNextbaseline(in_channels=4*F,
                           out_channels=1*F,
                           initial_dimension=42,
                           blocks_per_stage=2,
                           up_down_blocks=4,
                           bottleneck_blocks=1,
                           spatial_resolution=(256,256)).to(device)

In [17]:
# Losses combiner 

class AutomaticWeightedLoss(nn.Module):
    def __init__(self, num_losses):
        super (AutomaticWeightedLoss, self).__init__()
        params = torch.ones(num_losses, requires_grad = True)
        self.params = nn.Parameter(params)

    def forward(self, *losses):
        loss_sum = 0
        for i, loss in enumerate(losses):
            # Compute the wheighted loss component for each task
            weighted_loss = 0.5 / (self.params[i]**2) * loss

            # Add a regularisation term to encourage the learning of useful weights
            regularisation =torch.log(1 + self.params[i] **2)

            # Sum the weighted loss and the regularisation term
            loss_sum += weighted_loss + regularisation

        return loss_sum

In [18]:
#awl_combiner = AutomaticWeightedLoss(num_losses=5)
awl_combiner = AutomaticWeightedLoss(num_losses=4)

In [19]:
# Stablish the optimiser
lr_max = 5e-3

# Alone optimiser
optimizer_alone = torch.optim.AdamW(model_cnet.parameters(), lr=lr_max, weight_decay=1e-4)

# Including physics 
parameters = list(model_cnet.parameters()) + list(awl_combiner.parameters())
optimizer = torch.optim.AdamW(parameters, lr=lr_max, weight_decay=1e-4)

In [20]:
lr_max = 5e-3
optimizer = torch.optim.AdamW(model_cnet.parameters(), lr=lr_max, weight_decay=1e-4)

# Parameters for wrking the learning scheduler
warmup_epochs = 5
epochs = 10

steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * epochs

warmup_steps = steps_per_epoch*warmup_epochs
cosine_steps = total_steps - warmup_steps

# Inisialise the sheduler 
warmup_scheduler = LinearLR(optimizer=optimizer, start_factor=0.1, end_factor=1, total_iters=warmup_steps)
cosine_scheduler = CosineAnnealingLR(optimizer=optimizer, T_max = cosine_steps, eta_min = 1e-6)

scheduler_combined = SequentialLR(
                                optimizer=optimizer, 
                                schedulers=[warmup_scheduler, cosine_scheduler],
                                milestones=[warmup_steps])

In [21]:
# Parameters for working the learning scheduler
warmup_epochs = 5
epochs = 10

steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * epochs

warmup_steps = steps_per_epoch*warmup_epochs
cosine_steps = total_steps - warmup_steps

# Inisialise the sheduler 

# Scheduler for warmup epochs
warmup_alone_scheduler = LinearLR(optimizer=optimizer_alone, start_factor=0.1, end_factor=1, total_iters=warmup_steps)
cosine_alone_scheduler = CosineAnnealingLR(optimizer=optimizer_alone, T_max = cosine_steps, eta_min = 1e-6)
scheduler_alone = SequentialLR(
                            optimizer=optimizer_alone, 
                            schedulers=[warmup_alone_scheduler, cosine_alone_scheduler],
                            milestones=[warmup_steps])


# Scheduler for the epochs including physics
warmup_scheduler = LinearLR(optimizer=optimizer, start_factor=0.1, end_factor=1, total_iters=warmup_steps)
cosine_scheduler = CosineAnnealingLR(optimizer=optimizer, T_max = cosine_steps, eta_min = 1e-6)
scheduler_combined = SequentialLR(
                                optimizer=optimizer, 
                                schedulers=[warmup_scheduler, cosine_scheduler],
                                milestones=[warmup_steps])

In [22]:
losses_mse_train = []
losses_physics_train = []
losses_total_train = []

losses_mse_valid = []
losses_physics_valid = []
losses_total_valid = []

In [23]:
patience = 10
counter = 0
best_val_loss = float('inf')

In [24]:
for epoch in range(30):
    
    # Training 
    mse_train = 0
    physics_train = 0
    total_train = 0
    model_cnet.train()
    if epoch >= 10:
        awl_combiner.train()

    for batch in (bar := tqdm(train_loader, desc=f"Training Epoch {epoch}", bar_format='{l_bar}{bar}{r_bar}')):
        x = batch["input_fields"]
        x = x.to(device)
        x = preprocess(x)
        x = rearrange(x, "B Ti Lx Ly F -> B (Ti F) Lx Ly")

        y = batch["output_fields"]
        y = y.to(device)
        y = preprocess(y)
        y = rearrange(y, "B To Lx Ly F -> B (To F) Lx Ly")

        fx = model_cnet(x)
        #print(fx)

        # Check whether we have a NaN
        discriminant_nan = torch.isnan(fx).any().item()
        #print(discriminant_nan)

        if discriminant_nan == True:
            print(f"We have a NaN in fx")
            break
        mse = (fx - y).square().mean()
        
        
        ######### Physics losses
        # Concentration, concentration prev and concentration true
        rho = concentration_comp(fx)
        rho_true = concentration_comp(y)

        # Velocities pred and velocities true
        vx, vy = vel_comp(fx)
        vx_true, vy_true = vel_comp(y)

        # Partials of the velocities
        dvx_dx, _, _, dvy_dy = partials_velocities(fx)

        # Relevant components : orientation tensor D_xy D_yx and strain tensor E_xy E_yx
        _, D_xy, D_yx, _ = orientation_tensor_comp(fx)
        E_xx, E_xy, E_yx, E_yy = strain_tensor_comp(fx) 

        # Continuity equation -> Smoluchowski equation -> average of concentration field must be 1
        rho_average = rho.mean(dim=(1,2))
        continuity_cond = rho_average - torch.ones(rho_average.shape).to(device)
        loss_cont = continuity_cond.square().mean()

        # Incompresibility loss
        div_tmp = dvx_dx + dvy_dy
        loss_div =  div_tmp.square().mean()  

        # # Strain-velocity constraint
        # strain_xx = E_xx - (1/2)*(dvx_dx + dvx_dx)
        # strain_xy = E_xy - (1/2)*(dvx_dy + dvy_dx)
        # strain_yx = E_yx - (1/2)*(dvy_dx + dvx_dy)
        # strain_yy = E_yy - (1/2)*(dvy_dy + dvy_dy)
        # strain_tmp = strain_xx.square() + strain_xy.square() + strain_yx.square() + strain_yy.square()
        # loss_strain = strain_tmp.mean() 

        # Tensor symmetry
        sym_1 = D_xy - D_yx
        sym_2 = E_xy - E_yx
        loss_sym = sym_1.square().mean() + sym_2.square().mean() 


        # Global kinetic energy loss
        k_pred = (1/2)*rho*(vx**2 + vy**2)
        k_true = (1/2)*rho_true*(vx_true**2 + vy_true**2)
        k_global = k_pred - k_true

        loss_KE = k_global.square().mean()

        # Check whether we have a NaN
        discriminant_nan_cont = torch.isnan(loss_cont).any().item()
        discriminant_nan_div = torch.isnan(loss_div).any().item()
        #discriminant_nan_strain = torch.isnan(loss_strain).any().item()
        discriminant_nan_sym = torch.isnan(loss_sym).any().item()
        discriminant_nan_KE = torch.isnan(loss_KE).any().item()

        #if discriminant_nan==True or discriminant_nan_cont==True or discriminant_nan_div == True or discriminant_nan_strain==True or discriminant_nan_sym == True or discriminant_nan_KE==True:
        if discriminant_nan==True or discriminant_nan_cont==True or discriminant_nan_div == True or discriminant_nan_sym == True or discriminant_nan_KE==True:
            print(f"We have a NaN in mse or loss for physics")
            break

        ######################
        # Losses 
        # Warm up epochs
        if epoch < 10:
            total_loss = mse
        else:
            # Combinig losses
            #physics = awl_combiner(loss_cont, loss_div, loss_strain, loss_sym, loss_KE)
            physics = awl_combiner(loss_cont, loss_div, loss_sym, loss_KE)
            total_loss = mse + physics

        total_loss.backward()
        

        #To prevent NaN - exploiding gradients -> clipping gradient
        nn.utils.clip_grad_norm_(model_cnet.parameters(), max_norm = 1.0)
        
        if epoch < 10:
            optimizer_alone.step()
            optimizer_alone.zero_grad()
        else:
            optimizer.step()
            optimizer.zero_grad()

        if epoch < 10:
            bar.set_postfix(loss_mse=mse.detach().item(), loss_physics='----', total_loss=total_loss.detach().item())
        else:
            bar.set_postfix(loss_mse=mse.detach().item(), loss_physics=physics.detach().item(), total_loss=total_loss.detach().item())
            #print(f'loss_mse={mse},loss_div={loss_div}, loss_strain={loss_strain}, loss_sym={loss_sym}, physics_loss = {physics}')

        # Cumulative summation
        if epoch < 10 :
            mse_train += mse.detach().item()
        else:    
            mse_train += mse.detach().item()
            physics_train += physics.detach().item()
        total_train += total_loss.detach().item()

        # Update the learning rate at the end of each batch
        # Warmup epoch
        if epoch <10:
            scheduler_alone.step()
        # Including physics
        else:
            scheduler_combined.step()


    #if discriminant_nan==True or discriminant_nan_cont==True or discriminant_nan_div == True or discriminant_nan_strain==True or discriminant_nan_sym == True or discriminant_nan_KE==True:
    if discriminant_nan==True or discriminant_nan_cont==True or discriminant_nan_div == True or discriminant_nan_sym == True or discriminant_nan_KE==True:
        break

    mse_train/= max(1, len(train_loader))
    losses_mse_train.append(mse_train)

    if epoch < 10:
        losses_physics_train.append(np.nan)
    else:
        physics_train/= max(1, len(train_loader))
        losses_physics_train.append(physics_train)

    total_train/= max(1, len(train_loader))
    losses_total_train.append(total_train)

    # Validation (Evaluation)
    mse_valid = 0
    physics_valid = 0
    total_valid = 0
    model_cnet.eval()
    
    if epoch >= 10:
        awl_combiner.eval()
    
    with torch.no_grad():
        for batch in (bar := tqdm(valid_loader, desc=f"Validation Epoch {epoch}", bar_format='{l_bar}{bar}{r_bar}')):

            x_val_tmp = batch["input_fields"]
            x_val_tmp = x_val_tmp.to(device)
            x_val_tmp = preprocess(x_val_tmp)
            x_val_tmp = rearrange(x_val_tmp, "B Ti Lx Ly F -> B (Ti F) Lx Ly")

            y_val_tmp = batch["output_fields"]
            y_val_tmp = y_val_tmp.to(device)
            y_val_tmp = preprocess(y_val_tmp)
            y_val_tmp = rearrange(y_val_tmp, "B To Lx Ly F -> B (To F) Lx Ly")

            fx_val = model_cnet(x_val_tmp)

            mse_val = (fx_val - y_val_tmp).square().mean()

            ######### Physics losses
            # Concentration, concentration prev and concentration true
            rho_val = concentration_comp(fx_val)
            rho_true_val = concentration_comp(y_val_tmp)

            # Velocities pred and velocities true
            vx_val, vy_val = vel_comp(fx_val)
            vx_true_val, vy_true_val = vel_comp(y_val_tmp)

            #Partials of the velocities
            dvx_dx_val, _, _, dvy_dy_val = partials_velocities(fx_val)

            # Relevant components : orientation tensor D_xy D_yx and strain tensor E_xy E_yx
            _, D_xy_val, D_yx_val, _ = orientation_tensor_comp(fx_val)
            E_xx_val, E_xy_val, E_yx_val, E_yy_val = strain_tensor_comp(fx_val) 

            # Continuity equation -> Smoluchowski equation -> average of concentration field must be 1
            rho_average_val = rho_val.mean(dim=(1,2))
            continuity_cond_val = rho_average_val - torch.ones(rho_average_val.shape).to(device)
            loss_cont_val = continuity_cond_val.square().mean()

            # Incompresibility loss
            div_val_tmp = dvx_dx_val + dvy_dy_val
            loss_div_val = div_val_tmp.square().mean() #torch.linalg.norm(div_val_tmp**2)

            # # Strain-velocity constraint
            # strain_val_xx = E_xx_val - (1/2)*(dvx_dx_val + dvx_dx_val)
            # strain_val_xy = E_xy_val - (1/2)*(dvx_dy_val + dvy_dx_val)
            # strain_val_yx = E_yx_val - (1/2)*(dvy_dx_val + dvx_dy_val)
            # strain_val_yy = E_yy_val - (1/2)*(dvy_dy_val + dvy_dy_val)
            # strain_val_tmp = strain_val_xx.square() + strain_val_xy.square() + strain_val_yx.square() + strain_val_yy.square()
            # loss_strain_val = strain_val_tmp.mean() 

            # Tensor symmetry
            sym_1_val = D_xy_val - D_yx_val
            sym_2_val = E_xy_val - E_yx_val
            loss_sym_val = sym_1_val.square().mean() + sym_2_val.square().mean() 

            # Global kinetic energy loss
            k_pred_val = (1/2)*rho_val*(vx_val**2 + vy_val**2)
            k_true_val = (1/2)*rho_true_val*(vx_true_val**2 + vy_true_val**2)
            k_global_val = k_pred_val - k_true_val

            loss_KE_val = k_global_val.square().mean()

            ######################
            # Losses
            # Warmup epochs
            if epoch < 10:
                total_loss_val = mse_val
                bar.set_postfix(loss=mse_val.detach().item(), loss_physics="----", total=total_loss_val.detach().item())
            else:
                # Combinig losses
                #physics_loss_val =  awl_combiner(loss_cont_val, loss_div_val, loss_strain_val, loss_sym_val, loss_KE_val) 
                physics_loss_val =  awl_combiner(loss_cont_val, loss_div_val, loss_sym_val, loss_KE_val) 
                total_loss_val = mse_val + physics_loss_val            
                bar.set_postfix(loss=mse_val.detach().item(), loss_physics=physics_loss_val.detach().item(), total=total_loss_val.detach().item())

            # Cumulative summation
            if epoch < 10:
                mse_valid += mse_val.detach().item()
            else:
                mse_valid += mse_val.detach().item()
                physics_valid += physics_loss_val.detach().item()
            total_valid += total_loss_val.detach().item()
        
        
        mse_valid /= max(1, len(valid_loader))
        losses_mse_valid.append(mse_valid)

        if epoch < 10:
            losses_physics_valid.append(np.nan)
        else:
            physics_valid /= max(1, len(valid_loader))
            losses_physics_valid.append(physics_valid)

        total_valid /= max(1, len(valid_loader))
        losses_total_valid.append(total_valid)

    model_path = f'./model_UNetConvNext_phys_stats/trained_UNetConvNext_phys_epoch_{epoch}.pth'
    torch.save(model_cnet.state_dict(), model_path)

    # Early stopping condition
    if epoch >= 10 :
        if total_valid < best_val_loss:
            best_val_loss = total_valid
            counter=0
            
            model_path = f'./model_UNetConvNext_phys_stats/trained_UNetConvNext_phys_best.pth'
            torch.save(model_cnet.state_dict(), model_path)
            
        else:
            counter+=1
            
            if counter>=patience: 
                print("Early stopping triggered.")
                break
    
# PERFORM CLEANUP AFTER ALL EPOCHS ARE DONE
#del x, y, fx, loss_mse, batch, loss_mse_val

torch.cuda.empty_cache()

Training Epoch 0:   0%|          | 0/3369 [00:00<?, ?it/s]

Validation Epoch 29: 100%|██████████| 462/462 [02:22<00:00,  3.24it/s, loss=0.0745, loss_physics=2.83, total=2.91]


In [25]:
torch.cuda.empty_cache()

In [26]:
# Save the last model computed
model_path = './model_UNetConvNext_phys_stats/trained_UNetConvNext_phys_last_epoch.pth'
torch.save(model_cnet.state_dict(), model_path)

In [27]:
losses_mse_train = np.array(losses_mse_train)
losses_physics_train = np.array(losses_physics_train)
losses_total_train = np.array(losses_total_train)

losses_mse_valid = np.array(losses_mse_valid)
losses_physics_valid = np.array(losses_physics_valid)
losses_total_valid = np.array(losses_total_valid)

epochs = np.linspace(0, losses_mse_train.shape[0]-1, losses_mse_train.shape[0])

In [28]:
# Save the loss history in a DataFrame
df = pd.DataFrame({'Epochs': epochs ,'mse_train':losses_mse_train, 'loss_physics_train':losses_physics_train, \
                   'total_train':losses_total_train, 'mse_valid':losses_mse_valid, \
                    'loss_physics_valid': losses_physics_valid, 'total_valid':losses_total_valid})
print(df)
df.to_csv("7_loss_UNetConvNext_physics.csv", sep=',', float_format='{:.2e}'.format)

    Epochs  mse_train  loss_physics_train  total_train  mse_valid  \
0      0.0   0.070301                 NaN     0.070301   0.024392   
1      1.0   0.025036                 NaN     0.025036   0.011478   
2      2.0   0.021232                 NaN     0.021232   0.011860   
3      3.0   0.017930                 NaN     0.017930   0.012195   
4      4.0   0.017535                 NaN     0.017535   0.010879   
5      5.0   0.012405                 NaN     0.012405   0.007003   
6      6.0   0.008707                 NaN     0.008707   0.004185   
7      7.0   0.004724                 NaN     0.004724   0.003529   
8      8.0   0.002709                 NaN     0.002709   0.002338   
9      9.0   0.001981                 NaN     0.001981   0.002081   
10    10.0   0.066906            3.091409     3.158315   0.075704   
11    11.0   0.085481            3.214030     3.299511   0.083841   
12    12.0   0.094648            3.330404     3.425052   0.084767   
13    13.0   0.101681            3

Advices

Try to increase the number of workers to 4 again as we now don't have the condition of strain-rate (i.e. less computationally cost)

Try to increase the number of epochs to 40  